# Stage 4 — Numerical Analysis (Regression Models)

For each regressor in `REGRESSION_MODELS` × each seed in
`CONFIG["random_seeds"]`, fits and evaluates **two variants** of the same
model on the same chronological train/test split:

- **Baseline** — numerical features only (returns, volatility, drawdown,
  fundamental ratios, log assets).
- **+NLP** — baseline features plus rolling K-day mean class probabilities
  from the best Stage 3 NLP model (`artifacts/nlp_probs.parquet`).

## Why this comparison matters
The baseline-vs-+NLP delta on the same model, same seed, same split is the
thesis's primary signal-uplift question: **does sentiment add information
about future realized volatility, after controlling for price-based features?**
Reporting per-model, per-seed deltas lets us separate genuine NLP uplift from
seed-level noise, and Stage 6 averages these across seeds to give a single
headline R² improvement per regressor.

## What this stage produces
`./results/regression_results.csv` — one row per (model, variant, seed) with
R², RMSE, MAE, MSE, train/inference times, sample count, and ISO 8601 UTC
start/end timestamps for both phases (used by the external energy join).

## What this stage consumes
- `./artifacts/num_df.parquet` (Stage 1) — numerical features and risk_score target.
- `./artifacts/text_df.parquet` (Stage 1) — only used to compute the shared cutoff.
- `./artifacts/nlp_probs.parquet` (Stage 3) — per-headline class probabilities.

## Caveats worth disclosing in the writeup
- **Numerical test set is small (~46 rows).** This is a consequence of the
  FNSPID coverage limitation (AMZN headlines only span 2020-04 → 2023-12),
  which compresses the overlapping date range used for splits.
- **For dates with no headline coverage**, `attach_nlp_prob_features` falls
  back to neutral 1/3 priors, so the +NLP variant's NLP features are
  effectively constant for most pre-2020 rows.


In [1]:
# Load shared helpers/config plus model registries.
from common import *

# Regression metrics used for evaluation.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# inspect is used to check whether model factory expects a seed.
import inspect
# datetime is used to record wall-clock ISO timestamps for external energy join.
from datetime import datetime, timezone

# Load artifacts from previous stages.
text_df = load_text_df()
num_df = load_num_df()
# nlp_probs_df contains per-headline probabilities from Stage 3.
nlp_probs_df = load_nlp_probs_df()

# Normalize timestamps to date granularity for consistent splitting/joining.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# Shared chronological split boundary to align text and numeric evaluations.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)

# Build the +NLP numerical table by attaching rolling K-day mean sentiment
# probabilities to each numerical row. For dates with no headlines in the
# lookback window (most pre-2020 rows, given FNSPID's 2020-04+ AMZN coverage),
# this helper falls back to neutral 1/3 priors for each class. That means
# the +NLP variant for those rows has NLP features that are effectively
# constant — any uplift it shows comes entirely from the post-2020 segment.
num_aug_df = attach_nlp_prob_features(
    num_df=num_df,
    nlp_probs_df=nlp_probs_df,
    lookback_days=CONFIG["regression"]["nlp_lookback_days"],
    date_col="date",
)

# Collector for per-run records saved at end of stage.
results = ResultsCollector()

[common] device=cuda  artifacts=/home/hrilab/energy-analysis-pipeline/lab_machine_pkg/artifacts  results=/home/hrilab/energy-analysis-pipeline/lab_machine_pkg/results


/home/hrilab/energy-analysis-pipeline/lab_machine_pkg/.venv/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


## 4.1 Training routine

In [2]:
def train_regression_model(model_name, model_factory, X_tr, X_te, y_tr, y_te, seed):
    """Train one regression model for one seed and return metrics + fitted model."""

    # The REGRESSION_MODELS registry mixes factories of two shapes: sklearn
    # linear models that don't need a seed (LinearRegression, Ridge, Lasso,
    # ElasticNet are deterministic given the data) and gradient-boosted models
    # that do (XGBoost / LightGBM / CatBoost use random_state for column
    # subsampling and tie-breaking). Rather than maintain two parallel call
    # paths, we introspect the factory's signature: if it accepts any
    # arguments, pass `seed`; otherwise call it argument-less. This keeps the
    # registry definition trivial in common.py.
    sig = inspect.signature(model_factory)
    model = model_factory(seed) if sig.parameters else model_factory()

    # Measure fit time + wall-clock anchors for external energy join.
    wall_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    wall_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Measure prediction time + wall-clock anchors for external energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    y_pred = model.predict(X_te)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # mse is used directly and also converted to rmse.
    mse = mean_squared_error(y_te, y_pred)
    return {
        "model": model_name,
        "seed": seed,
        "mse": mse,
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_te, y_pred),
        "r2": r2_score(y_te, y_pred),
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(y_te),
        "wall_train_start_iso": wall_train_start_iso,
        "wall_train_end_iso": wall_train_end_iso,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }, model

## 4.2 Run every (model, seed) × (baseline, +NLP) combination

In [3]:
# trained_reg_models: built but never persisted and never read downstream.
# Kept for optional in-notebook diagnostics (e.g., feature-importance plots
# on a specific tree-based regressor). Safe to remove if unused.
trained_reg_models = {}

# Baseline numerical split (no NLP features).
(
    X_tr_base,
    X_te_base,
    y_tr,
    y_te,
    _,
    base_feature_cols,
    _,
    _,
) = make_num_splits_chronological(
    df=num_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Augmented numerical split (includes rolling NLP probabilities).
(
    X_tr_aug,
    X_te_aug,
    y_tr_aug,
    y_te_aug,
    _,
    aug_feature_cols,
    _,
    _,
) = make_num_splits_chronological(
    df=num_aug_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Defensive check. If at any point baseline and augmented splits started
# disagreeing on row ordering or target values (e.g., because a future
# refactor introduced a different sort key inside one of the split helpers),
# the per-row baseline-vs-+NLP comparison would become meaningless: the same
# (model, seed) would be predicting on different rows in each variant. This
# guard catches that class of bug at runtime rather than letting it silently
# corrupt the uplift table.
if not np.array_equal(y_tr, y_tr_aug) or not np.array_equal(y_te, y_te_aug):
    raise ValueError("Baseline and +NLP splits do not align on targets.")

print(f"Baseline feature count: {len(base_feature_cols)}")
print(f"Augmented feature count: {len(aug_feature_cols)}")

# Train every model for every seed for baseline and +NLP variants.
for model_name, factory in REGRESSION_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        baseline_record, baseline_model = train_regression_model(
            model_name=model_name,
            model_factory=factory,
            X_tr=X_tr_base,
            X_te=X_te_base,
            y_tr=y_tr,
            y_te=y_te,
            seed=seed,
        )
        results.add_regression(baseline_record)
        trained_reg_models[(model_name, seed, "baseline")] = baseline_model

        augmented_record, augmented_model = train_regression_model(
            model_name=f"{model_name} +NLP",
            model_factory=factory,
            X_tr=X_tr_aug,
            X_te=X_te_aug,
            y_tr=y_tr,
            y_te=y_te,
            seed=seed,
        )
        results.add_regression(augmented_record)
        trained_reg_models[(model_name, seed, "+NLP")] = augmented_model

        # Print side-by-side R² so uplift is easy to inspect while running.
        print(
            f"{model_name} (seed {seed})  "
            f"baseline R²={baseline_record['r2']:.4f}, "
            f"+NLP R²={augmented_record['r2']:.4f}"
        )

Baseline feature count: 11
Augmented feature count: 14
Linear Regression (seed 0)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 1)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 2)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 3)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 4)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 5)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 6)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 7)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 8)  baseline R²=-0.1138, +NLP R²=-0.0187
Linear Regression (seed 9)  baseline R²=-0.1138, +NLP R²=-0.0187
Ridge (seed 0)  baseline R²=-0.1135, +NLP R²=-0.0540
Ridge (seed 1)  baseline R²=-0.1135, +NLP R²=-0.0540
Ridge (seed 2)  baseline R²=-0.1135, +NLP R²=-0.0540
Ridge (seed 3)  baseline R²=-0.1135, +NLP R²=-0.0540
Ridge (seed 4)  baseline R²=-0.1135, +NLP R²=-0.0540
Ridge (seed 5)  baseline R²=-0

XGBoost (seed 0)  baseline R²=0.0958, +NLP R²=0.2805
XGBoost (seed 1)  baseline R²=0.0958, +NLP R²=0.2805
XGBoost (seed 2)  baseline R²=0.0958, +NLP R²=0.2805


XGBoost (seed 3)  baseline R²=0.0958, +NLP R²=0.2805
XGBoost (seed 4)  baseline R²=0.0958, +NLP R²=0.2805
XGBoost (seed 5)  baseline R²=0.0958, +NLP R²=0.2805


XGBoost (seed 6)  baseline R²=0.0958, +NLP R²=0.2805
XGBoost (seed 7)  baseline R²=0.0958, +NLP R²=0.2805
XGBoost (seed 8)  baseline R²=0.0958, +NLP R²=0.2805


XGBoost (seed 9)  baseline R²=0.0958, +NLP R²=0.2805
LightGBM (seed 0)  baseline R²=0.0472, +NLP R²=-0.0086


...

CatBoost (seed 0)  baseline R²=0.4154, +NLP R²=0.1983
CatBoost (seed 1)  baseline R²=0.3275, +NLP R²=0.1650


CatBoost (seed 2)  baseline R²=0.3119, +NLP R²=0.1237
CatBoost (seed 3)  baseline R²=0.2248, +NLP R²=0.3945


CatBoost (seed 4)  baseline R²=0.3418, +NLP R²=0.4527
CatBoost (seed 5)  baseline R²=0.2144, +NLP R²=0.4781


CatBoost (seed 6)  baseline R²=0.4666, +NLP R²=0.3110
CatBoost (seed 7)  baseline R²=0.2273, +NLP R²=0.4291


CatBoost (seed 8)  baseline R²=0.3837, +NLP R²=0.4586
CatBoost (seed 9)  baseline R²=0.2345, +NLP R²=0.3114


## 4.3 Baseline vs +NLP comparison

In [4]:
reg_df = results.regression_df()
# Mean baseline R² by base model name.
baseline_mean = reg_df[~reg_df["model"].str.endswith("+NLP")].groupby("model")["r2"].mean()
# Mean augmented R² by base model name (remove suffix for alignment).
aug_mean = reg_df[reg_df["model"].str.endswith("+NLP")].assign(
    base_model=lambda d: d["model"].str.replace(" +NLP", "", regex=False)
).groupby("base_model")["r2"].mean()
comparison = pd.concat(
    [baseline_mean.rename("R²_baseline"), aug_mean.rename("R²_+NLP")], axis=1
)
comparison["R²_improvement"] = comparison["R²_+NLP"] - comparison["R²_baseline"]
print("=== Baseline vs NLP-Augmented R² (mean over seeds) ===")
display(comparison.round(6))
reg_df.round(6)

=== Baseline vs NLP-Augmented R² (mean over seeds) ===


,R²_baseline,R²_+NLP,R²_improvement
CatBoost,0.314787,0.332257,0.017470
Elastic Net,-0.000873,-0.000873,0.000000
Lasso,-0.000873,-0.000873,0.000000
LightGBM,0.047223,-0.008615,-0.055838
Linear Regression,-0.113846,-0.018675,0.095171
Ridge,-0.113454,-0.054047,0.059408
XGBoost,0.095785,0.280457,0.184672


,model,seed,mse,rmse,mae,r2,train_time_s,infer_time_s,n_test_samples,wall_train_start_iso,wall_train_end_iso,wall_infer_start_iso,wall_infer_end_iso
0,Linear Regression,0,0.025741,0.160439,0.136401,-0.113846,0.000596,0.000052,46,2026-05-06T21:30:05.332774+00:00,2026-05-06T21:30:05.333376+00:00,2026-05-06T21:30:05.333382+00:00,2026-05-06T21:30:05.333436+00:00
1,Linear Regression +NLP,0,0.023541,0.153432,0.136750,-0.018675,0.000487,0.000045,46,2026-05-06T21:30:05.333846+00:00,2026-05-06T21:30:05.334337+00:00,2026-05-06T21:30:05.334340+00:00,2026-05-06T21:30:05.334387+00:00
2,Linear Regression,1,0.025741,0.160439,0.136401,-0.113846,0.000375,0.000041,46,2026-05-06T21:30:05.334779+00:00,2026-05-06T21:30:05.335157+00:00,2026-05-06T21:30:05.335161+00:00,2026-05-06T21:30:05.335204+00:00
3,Linear Regression +NLP,1,0.023541,0.153432,0.136750,-0.018675,0.000416,0.000040,46,2026-05-06T21:30:05.335575+00:00,2026-05-06T21:30:05.335995+00:00,2026-05-06T21:30:05.335998+00:00,2026-05-06T21:30:05.336039+00:00
4,Linear Regression,2,0.025741,0.160439,0.136401,-0.113846,0.000369,0.000041,46,2026-05-06T21:30:05.336409+00:00,2026-05-06T21:30:05.336782+00:00,2026-05-06T21:30:05.336785+00:00,2026-05-06T21:30:05.336827+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,CatBoost +NLP,7,0.013193,0.114860,0.108990,0.429122,0.057635,0.000262,46,2026-05-06T21:30:07.993124+00:00,2026-05-06T21:30:08.050765+00:00,2026-05-06T21:30:08.050771+00:00,2026-05-06T21:30:08.051035+00:00
136,CatBoost,8,0.014243,0.119343,0.110371,0.383693,0.068176,0.000261,46,2026-05-06T21:30:08.051654+00:00,2026-05-06T21:30:08.119837+00:00,2026-05-06T21:30:08.119843+00:00,2026-05-06T21:30:08.120106+00:00
137,CatBoost +NLP,8,0.012511,0.111854,0.102280,0.458618,0.052987,0.000273,46,2026-05-06T21:30:08.120698+00:00,2026-05-06T21:30:08.173691+00:00,2026-05-06T21:30:08.173698+00:00,2026-05-06T21:30:08.173973+00:00
138,CatBoost,9,0.017690,0.133004,0.124417,0.234526,0.057077,0.000281,46,2026-05-06T21:30:08.174599+00:00,2026-05-06T21:30:08.231682+00:00,2026-05-06T21:30:08.231689+00:00,2026-05-06T21:30:08.231971+00:00


## 4.4 Persist results

In [5]:
results.save(RESULTS_DIR)
print(f"Total regression experiments: {len(results.regression_results)}")

Results saved to /home/hrilab/energy-analysis-pipeline/lab_machine_pkg/results/
Total regression experiments: 140
